In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv('/content/WA_Fn-UseC_-Telco-Customer-Churn.csv')

df['TotalCharges'] = df['TotalCharges'].replace(" ", np.nan)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'])
df['T_MODE'] = df['TotalCharges'].fillna(df['TotalCharges'].mode()[0])
df = df.drop('TotalCharges', axis=1)

df['Churn'] = df['Churn'].map({'Yes':1, 'No':0}).astype(int)

In [ ]:
# Add SIM column using your mapping logic
def assign_indian_operator(row):
    if row['InternetService'] == 'Fiber optic':
        return 'Reliance Jio'
    elif row['InternetService'] == 'DSL':
        return 'Airtel'
    elif row['PhoneService'] == 'Yes' and row['MultipleLines'] == 'Yes':
        return 'Vodafone Idea'
    else:
        return 'BSNL'

df['Sim'] = df.apply(assign_indian_operator, axis=1)

In [ ]:

from sklearn.model_selection import train_test_split

X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=45
)

X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

In [ ]:
X_train_nums = X_train.select_dtypes(exclude=object)
X_train_cats = X_train.select_dtypes(include=object)

X_test_nums = X_test.select_dtypes(exclude=object)
X_test_cats = X_test.select_dtypes(include=object)

In [ ]:

X_train_cats = X_train_cats.drop(['customerID'], axis=1)
X_test_cats = X_test_cats.drop(['customerID'], axis=1)



In [ ]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(drop='first')

simple_cols = ['gender','Partner','Dependents']

ohe.fit(X_train_cats[simple_cols])

train_ohe = pd.DataFrame(ohe.transform(X_train_cats[simple_cols]).toarray(),
                         columns=ohe.get_feature_names_out())

test_ohe = pd.DataFrame(ohe.transform(X_test_cats[simple_cols]).toarray(),
                        columns=ohe.get_feature_names_out())

train_ohe.reset_index(drop=True, inplace=True)
test_ohe.reset_index(drop=True, inplace=True)

X_train_cats = pd.concat([X_train_cats, train_ohe], axis=1)
X_test_cats = pd.concat([X_test_cats, test_ohe], axis=1)

X_train_cats = X_train_cats.drop(simple_cols, axis=1)
X_test_cats = X_test_cats.drop(simple_cols, axis=1)

In [ ]:

from sklearn.preprocessing import OrdinalEncoder

ord_cols = ['PhoneService','MultipleLines','InternetService','OnlineSecurity',
            'OnlineBackup','DeviceProtection','TechSupport','StreamingTV',
            'StreamingMovies','Contract','PaperlessBilling','PaymentMethod','Sim']

oe = OrdinalEncoder()
oe.fit(X_train_cats[ord_cols])

train_ord = pd.DataFrame(oe.transform(X_train_cats[ord_cols]),
                         columns=[col+'_OD' for col in ord_cols])

test_ord = pd.DataFrame(oe.transform(X_test_cats[ord_cols]),
                        columns=[col+'_OD' for col in ord_cols])

train_ord.reset_index(drop=True,inplace=True)
test_ord.reset_index(drop=True,inplace=True)

X_train_cats = pd.concat([X_train_cats, train_ord], axis=1)
X_test_cats = pd.concat([X_test_cats, test_ord], axis=1)

X_train_cats = X_train_cats.drop(ord_cols, axis=1)
X_test_cats = X_test_cats.drop(ord_cols, axis=1)

X_train_nums = X_train_nums.reset_index(drop=True)
X_train_cats = X_train_cats.reset_index(drop=True)

In [ ]:


Training_data = pd.concat([X_train_nums, X_train_cats], axis=1)

X_test_nums = X_test_nums.reset_index(drop=True)
X_test_cats = X_test_cats.reset_index(drop=True)

Testing_data = pd.concat([X_test_nums, X_test_cats], axis=1)

print("FINAL TRAIN DATA SHAPE:", Training_data.shape)
print("FINAL y_train SHAPE:", y_train.shape)



FINAL TRAIN DATA SHAPE: (5634, 20)
FINAL y_train SHAPE: (5634,)


In [ ]:
# Random over sampling
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(sampling_strategy=1.0, random_state=42)
Training_data_resample, y_train_resample = ros.fit_resample(Training_data, y_train)

print("AFTER OVERSAMPLING:", Training_data_resample.shape, y_train_resample.shape)

AFTER OVERSAMPLING: (8236, 20) (8236,)


In [ ]:
Training_data_resample.sample(10)

,SeniorCitizen,tenure,MonthlyCharges,T_MODE,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_OD,MultipleLines_OD,InternetService_OD,OnlineSecurity_OD,OnlineBackup_OD,DeviceProtection_OD,TechSupport_OD,StreamingTV_OD,StreamingMovies_OD,Contract_OD,PaperlessBilling_OD,PaymentMethod_OD,Sim_OD
7368,0,1,70.50,70.50,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,2.0
2336,1,1,69.65,69.65,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,2.0
2407,0,49,49.65,2409.90,1.0,1.0,0.0,0.0,1.0,0.0,0.0,2.0,0.0,0.0,2.0,2.0,2.0,0.0,1.0,0.0
4085,0,8,19.50,159.35,0.0,1.0,1.0,1.0,0.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,0.0,0.0,1.0
986,0,2,20.00,40.90,0.0,0.0,0.0,1.0,0.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,3.0,1.0
3204,0,8,94.75,759.55,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,2.0,0.0,2.0,2.0,0.0,1.0,2.0,2.0
239,1,37,106.75,4056.75,0.0,0.0,0.0,1.0,2.0,1.0,2.0,2.0,0.0,0.0,2.0,2.0,0.0,1.0,0.0,2.0
8095,1,33,94.50,3105.55,1.0,1.0,0.0,1.0,2.0,1.0,0.0,0.0,0.0,0.0,2.0,2.0,0.0,1.0,2.0,2.0
4174,0,1,18.85,18.85,0.0,0.0,0.0,1.0,0.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,2.0,1.0
3283,0,36,84.75,3050.15,1.0,0.0,0.0,1.0,2.0,1.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,1.0,2.0,2.0


In [ ]:
# feature Scaling
# Z-score

import sklearn

In [ ]:
from sklearn.preprocessing import MinMaxScaler

obj = MinMaxScaler()

obj.fit(Training_data_resample)

MinMaxScaler()

In [ ]:
Training_data_minmax = obj.transform(Training_data_resample)

In [ ]:
Testing_data_minmax = obj.transform(Testing_data)

In [ ]:
Testing_data_minmax

array([[1.        , 0.48611111, 0.92089552, ..., 1.        , 0.        ,
        0.66666667],
       [0.        , 0.18055556, 0.12537313, ..., 1.        , 0.        ,
        0.        ],
       [0.        , 0.84722222, 0.06716418, ..., 1.        , 0.33333333,
        1.        ],
       ...,
       [0.        , 0.68055556, 0.76766169, ..., 1.        , 0.        ,
        0.66666667],
       [0.        , 0.43055556, 0.71293532, ..., 1.        , 0.66666667,
        0.66666667],
       [0.        , 0.01388889, 0.64825871, ..., 1.        , 1.        ,
        0.66666667]])

In [ ]:
from sklearn.preprocessing import StandardScaler

obje = StandardScaler()

obje.fit(Training_data_resample)

StandardScaler()

In [ ]:
Training_data_Zscore = obje.transform(Training_data_resample)

In [ ]:
Training_data_Zscore

array([[-0.48014515,  1.39125839, -0.27563848, ..., -1.33754555,
         0.35331134, -1.35048747],
       [-0.48014515, -0.19745167,  0.03417468, ...,  0.74763809,
        -1.59962301, -1.35048747],
       [-0.48014515, -1.03361487, -0.80180162, ..., -1.33754555,
         0.35331134, -1.35048747],
       ...,
       [-0.48014515, -0.61553327,  0.45302823, ...,  0.74763809,
         0.35331134,  0.76347524],
       [-0.48014515, -0.94999855,  0.61053101, ...,  0.74763809,
        -0.62315584,  0.76347524],
       [-0.48014515, -1.11723119,  0.88399738, ...,  0.74763809,
         0.35331134,  0.76347524]])

In [ ]:
Testing_data_Zscore = obje.transform(Testing_data)

In [ ]:
Testing_data_Zscore

array([[ 2.08270351,  0.30424624,  1.48977731, ...,  0.74763809,
        -1.59962301,  0.76347524],
       [-0.48014515, -0.61553327, -1.27777157, ...,  0.74763809,
        -1.59962301, -1.35048747],
       [-0.48014515,  1.39125839, -1.48027515, ...,  0.74763809,
        -0.62315584,  1.82045659],
       ...,
       [-0.48014515,  0.88956048,  0.95669097, ...,  0.74763809,
        -1.59962301,  0.76347524],
       [-0.48014515,  0.1370136 ,  0.766303  , ...,  0.74763809,
         0.35331134,  0.76347524],
       [-0.48014515, -1.11723119,  0.54129902, ...,  0.74763809,
         1.32977852,  0.76347524]])